In [0]:
import os
import mlflow
import mlflow.spark
from pyspark.sql import functions as F
from pyspark.ml.functions import vector_to_array


mlflow.set_registry_uri("databricks-uc")
dbutils.fs.mkdirs("dbfs:/Volumes/workspace/default/mlops_project/mlflow_tmp")
os.environ["MLFLOW_DFS_TMP"] = "/Volumes/workspace/default/mlops_project/mlflow_tmp"
# UC temp dir (recommended for serverless/shared)
#dbutils.fs.mkdirs("dbfs:/Volumes/workspace/default/mlops_project/mlflow_tmp")
#os.environ["MLFLOW_DFS_TMP"] = "/Volumes/workspace/default/mlops_project/mlflow_tmp"

MODEL_NAME = "lr_end_to_end_pipeline_model"
MODEL_URI = f"models:/{MODEL_NAME}@Champion"   # use Production for jobs
# If you haven't promoted yet, temporarily use:
# MODEL_URI = f"models:/{MODEL_NAME}/latest"

INPUT_TABLE = "mlops_project.lendingclub_silver"
OUTPUT_TABLE = "mlops_project.lendingclub_gold_predictions_monitor"



In [0]:

model = mlflow.spark.load_model(MODEL_URI)
df = spark.table(INPUT_TABLE)



/databricks/python/lib/python3.12/site-packages/databricks/sdk/errors/base.py:87: UserWarning: The 'retry_after_secs' parameter of DatabricksError is deprecated and will be removed in a future version.
  warnings.warn(


In [0]:
scored = (
    model.transform(df)
       .withColumn("default_proba", vector_to_array("probability")[1])
         .withColumn("prediction_ts", F.current_timestamp())
         .withColumn("score_date", F.current_date())
         .withColumn("model_name", F.lit(MODEL_NAME))
         .withColumn("model_alias", F.lit("Champion"))
)

scored.select("score_date","prediction_ts","model_name","model_alias","default_proba","prediction","label_default").show(5, truncate=False)


+----------+--------------------------+----------------------------+-----------+-------------------+----------+-------------+
|score_date|prediction_ts             |model_name                  |model_alias|default_proba      |prediction|label_default|
+----------+--------------------------+----------------------------+-----------+-------------------+----------+-------------+
|2026-01-08|2026-01-08 15:29:07.381102|lr_end_to_end_pipeline_model|Champion   |0.41650284399967885|0.0       |0            |
|2026-01-08|2026-01-08 15:29:07.381102|lr_end_to_end_pipeline_model|Champion   |0.7311979618069673 |1.0       |0            |
|2026-01-08|2026-01-08 15:29:07.381102|lr_end_to_end_pipeline_model|Champion   |0.05632086112058621|0.0       |0            |
|2026-01-08|2026-01-08 15:29:07.381102|lr_end_to_end_pipeline_model|Champion   |0.3466768394002625 |0.0       |0            |
|2026-01-08|2026-01-08 15:29:07.381102|lr_end_to_end_pipeline_model|Champion   |0.5156251698159469 |1.0       |0      

In [0]:
OUTPUT_TABLE = "mlops_project.lendingclub_gold_predictions_monitor"


(scored
 .select("score_date","prediction_ts","model_name","model_alias","default_proba","prediction","probability","label_default")
 .write.format("delta")
 .mode("append")          # <-- important (keep history)
 .saveAsTable(OUTPUT_TABLE))

print("Gold rows:", spark.table(OUTPUT_TABLE).count())

Gold rows: 1329272
